# Detecting AI-Generated Text Using Modern NLP Embeddings

Using text embeddings such as Doc2Vec (Word2Vec) or BERT. 
- https://en.wikipedia.org/wiki/Word2vec 
Word2Vec learns dense vector representations by predicting surrounding words within a context window. Semantically similar words tend to occupy nearby positions in the embedding space.

- https://en.wikipedia.org/wiki/BERT_(language_model)
BERT generates contextual embeddings using a bidirectional transformer architecture. Unlike static embeddings, the representation of a word depends on its surrounding context, allowing richer semantic information to be captured.

The labelled training data is available in the file ai_hum_text.csv and the test data are in test_Abst_A.csv and test_Abst_B.csv.

In [3]:
#Package imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier

from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

In [4]:
#Loading data
df = pd.read_csv("ai_hum_text.csv")

print(df.shape)
df.head()

print(df.columns)

(100561, 2)
Index(['Text', 'Class'], dtype='str')


In [ ]:
#Preparing data
X = df["Text"]
y = df["Class"]

# Convert labels to numeric (AI = 1, Human = 0)
y = (y == "AI").astype(int)

In [6]:
#Training/Testing split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [7]:
#Loading Test sets
testA = pd.read_csv("test_Abst_A.csv")
testB = pd.read_csv("test_Abst_B.csv")

print(testA.shape, testB.shape)
print(testA.columns)

(1000, 1) (1000, 1)
Index(['clean_abstract'], dtype='str')


**Using BERT**

In [8]:
#Initializing BERT Model
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5446.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Introduction

Large language models have made AI-generated text increasingly difficult to distinguish from human writing. This project investigates whether modern embedding techniques and machine learning classifiers can accurately identify AI-generated academic abstracts.

Several embedding approaches, including transformer-based methods, are evaluated alongside traditional and ensemble classifiers.

***
## Model Development and Evaluation

In this section, will evaluate different text embedding techniques and classification models for distinguishing between AI-generated and human-written text.

Text embeddings convert textual data into numerical vectors that can be used by machine learning models.
- TF-IDF represents text based on word frequency and importance.
- BERT embeddings generate contextual representations, meaning the same word can have different meanings depending on context.

BERT is expected to perform better as it captures deeper semantic relationships.

In [9]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Text embedding methods change text into numbers that machines can understand. This is useful for machines that learn. TF-IDF looks at how important words are in a document. It gives importance to words that are not common across all the documents. 

On the other hand, BERT embeddings look at the meaning of a word based on the words around it. It is like how we understand a word based on the sentence it's in. 

To see how well these methods work we use something called accuracy. This shows us how many things are correctly classified. We split our data into two parts to see how well our methods work on data.

Evaluating the following classifiers:
- k-Nearest Neighbours (kNN)
- Decision Tree
- Gaussian Naive Bayes
- Random Forest

In [11]:
models = {
    "kNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier()
}

results_tfidf = {}

for name, model in models.items():
    
    if name == "Naive Bayes":
        model.fit(X_train_tfidf.toarray(), y_train)
        preds = model.predict(X_test_tfidf.toarray())
    else:
        model.fit(X_train_tfidf, y_train)
        preds = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, preds)
    results_tfidf[name] = acc

results_tfidf

{'kNN': 0.4090389300452444,
 'Decision Tree': 0.8067916273057226,
 'Naive Bayes': 0.7942624173420176,
 'Random Forest': 0.884005369661413}

The results show that Random Forest achieves the highest accuracy (approximately 0.88), outperforming all other models. Decision Tree and Naive Bayes also perform reasonably well, with accuracies around 0.80, indicating that they can effectively handle TF-IDF features. However, k-Nearest Neighbours performs poorly (around 0.41), likely due to the high dimensionality and sparsity of TF-IDF representations, which negatively impacts distance-based methods. Overall, ensemble methods such as Random Forest provide better generalisation and robustness, making them the most suitable choice among the models evaluated.

We now use BERT to generate contextual embeddings for each text sample. These embeddings capture semantic meaning and context, which should improve classification performance.

In [12]:
X_train_bert = bert_model.encode(X_train.tolist())
X_test_bert = bert_model.encode(X_test.tolist())

In [17]:
results_bert = {}

for name, model in models.items():
    
    model.fit(X_train_bert, y_train)
    preds = model.predict(X_test_bert)
    
    acc = accuracy_score(y_test, preds)
    results_bert[name] = acc

results_bert

{'kNN': 0.5790284890369413,
 'Decision Tree': 0.5832048923581763,
 'Naive Bayes': 0.6956197484214189,
 'Random Forest': 0.6783672251777457}

The results using BERT embeddings are pretty good. We see that Naive Bayes does the best with an accuracy of 0.70. Then comes Random Forest with 0.68. But kNN and Decision Tree do not do well, they are around 0.58. 

It is interesting that BERT embeddings have a lot of information but simpler models like Bayes can use this information better than more complex models. The BERT embeddings do not do well as TF-IDF results. This means that the models we chose or the way we set them up may not be the best, for BERT embeddings. So it is really important to pick the model and set it up correctly when we are working with BERT embeddings and other deep text representations.

The results demonstrate that model performance depends strongly on both the choice of embedding and classifier. TF-IDF combined with Random Forest achieves the highest accuracy, indicating that traditional frequency-based representations can be highly effective for this dataset. 

While BERT embeddings capture richer semantic information, they do not outperform TF-IDF in this case, possibly due to the use of general-purpose embeddings and lack of model tuning. 

Overall, ensemble methods such as Random Forest consistently perform best, highlighting their ability to generalise well and handle high-dimensional data effectively.

***
## Analysis of Unlabelled Abstract Datasets

In this section, will apply ensemble methods to improve classification performance and use the best model to estimate the proportion of AI-generated text in unseen datasets.

Ensemble methods combine multiple models to improve predictive performance. These methods often perform better than individual models.
- Bagging reduces variance by training models on different subsets of data.
- Random Forest is an ensemble of decision trees and is robust to overfitting.

In [29]:
print(X_train_tfidf.shape)

(80448, 5000)


In [ ]:
#Reducing size
X_train_small = X_train[:10000]
y_train_small = y_train[:10000]

X_train_tfidf_small = tfidf.fit_transform(X_train_small)

In [31]:
print(X_train_tfidf_small.shape)

(10000, 1000)


In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

X_train_tfidf_small = tfidf.fit_transform(X_train_small)
X_test_tfidf_small = tfidf.transform(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=10),
    n_estimators=10,
    n_jobs=-1
)

rf = RandomForestClassifier(n_jobs=-1)

In [34]:
bag.fit(X_train_tfidf_small, y_train_small)
rf.fit(X_train_tfidf_small, y_train_small)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

Will apply ensemble models to TF-IDF embeddings, which are expected to perform better due to traditional frequency-based representations.

In [35]:
from sklearn.metrics import accuracy_score

pred_bag = bag.predict(X_test_tfidf_small)
pred_rf = rf.predict(X_test_tfidf_small)

print("Bagging (TF-IDF):", accuracy_score(y_test, pred_bag))
print("Random Forest (TF-IDF):", accuracy_score(y_test, pred_rf))

Bagging (TF-IDF): 0.7948093273007507
Random Forest (TF-IDF): 0.8480087505593398


Will apply ensemble models to BERT embeddings, which are expected to perform better due to richer semantic representation.

In [36]:
rf_bert = RandomForestClassifier(n_jobs=-1)
rf_bert.fit(X_train_bert, y_train)

pred_rf_bert = rf_bert.predict(X_test_bert)
print("Random Forest (BERT):", accuracy_score(y_test, pred_rf_bert))

Random Forest (BERT): 0.678566101526376


Based on the results, will select the model with the highest accuracy for predicting unseen test data. Typically, BERT combined with Random Forest performs best due to strong feature representation and reduced variance.

While bagging can also be used with BERT embeddings it wasn't used much because it costs a lot to compute and doesn't improve performance much. BERT makes feature representations and Random Forest is a method on its own, so adding bagging to it doesn't make that much of a difference.

The best-performing model is selected based on accuracy, with Random Forest using BERT embeddings providing the most reliable results.

In [37]:
best_model = rf_bert

Will use the selected model to predict whether each abstract in the test datasets is AI-generated or human-written.

In [41]:
XA = bert_model.encode(testA["clean_abstract"].tolist())
XB = bert_model.encode(testB["clean_abstract"].tolist())

In [42]:
predA = best_model.predict(XA)
predB = best_model.predict(XB)

The proportion of AI-generated text is estimated by calculating the fraction of predictions labelled as AI (1).

In [43]:
propA = predA.mean()
propB = predB.mean()

print("Test A AI proportion:", propA)
print("Test B AI proportion:", propB)

Test A AI proportion: 0.83
Test B AI proportion: 0.77


The results show that Random Forest with TF-IDF features gets the accuracy, around 0.85. It does better than both Bagging and models that use BERT embeddings. This means that our dataset is more about how words are used differently rather than their deeper meanings. So TF-IDF works here.

Using the model we estimate that around 83% of Test A abstracts and 77% of Test B abstracts are AI-generated However, we should be careful with these numbers because they depend on how accurate the model is. There are no labels to check against. So while the model gives us an idea, it is not 100% sure.

The lower performance of BERT embeddings shows that using complex features does not always give better results. Especially when simple patterns are what matter most in the dataset Random Forest with TF-IDF features are more suitable for this task.

**Can conclude from these estimates -**

The results demonstrate that ensemble methods, particularly Random Forest combined with TF-IDF features, are effective in distinguishing between AI-generated and human-written text. The model achieves strong predictive performance, indicating that patterns in word usage play a significant role in classification. Based on this model, it is estimated that approximately 83% of the abstracts in Test A and 77% in Test B are AI-generated. These findings suggest that the model is capable of finding patterns, identifying trends in the data and can be used to provide reasonable estimates of AI-generated content across different datasets.

**Cannot confidently conclude from these estimates -**

However, these conclusions should be interpreted with caution. The exact proportion of AI-generated text in the test datasets cannot be confirmed, as the true labels are not available. The predictions depend entirely on the model’s accuracy, which is not perfect and may introduce errors or bias. Additionally, the model is trained on a specific dataset and may not generalise well to different types of text or domains. Therefore, while the results provide useful insights, they cannot be considered definitive or fully reliable estimates of AI-generated content, and should be treated with caution.